In [0]:
%sql
CREATE OR REPLACE TABLE datasheet_project.silver.datasheets_quarantine AS
SELECT
  *,
  -- Inverse logic: TRUE if ANY rule fails
  NOT(
    chip_name IS NOT NULL
    AND (flash_kb IS NULL OR flash_kb > 0)
    AND (ram_kb IS NULL OR ram_kb > 0)
    AND (ram_kb IS NULL OR flash_kb IS NULL OR ram_kb <= flash_kb * 10)
    AND (voltage_min IS NULL OR voltage_max IS NULL OR voltage_min < voltage_max)
    AND (max_clock_mhz IS NULL OR max_clock_mhz BETWEEN 1 AND 1000)
    AND (pin_count IS NULL OR pin_count BETWEEN 4 AND 500)
  ) AS is_quarantined,
  
  -- Human-readable reason (first failure wins)
  CASE
    WHEN chip_name IS NULL THEN 'missing_chip_name'
    WHEN flash_kb IS NOT NULL AND flash_kb <= 0 THEN 'invalid_flash'
    WHEN ram_kb IS NOT NULL AND ram_kb <= 0 THEN 'invalid_ram'
    WHEN ram_kb IS NOT NULL AND flash_kb IS NOT NULL AND ram_kb > flash_kb * 10 THEN 'suspicious_ram_vs_flash'
    WHEN voltage_min IS NOT NULL AND voltage_max IS NOT NULL AND voltage_min >= voltage_max THEN 'invalid_voltage_range'
    WHEN max_clock_mhz IS NOT NULL AND (max_clock_mhz < 1 OR max_clock_mhz > 1000) THEN 'invalid_clock'
    WHEN pin_count IS NOT NULL AND (pin_count < 4 OR pin_count > 500) THEN 'invalid_pins'
    ELSE NULL
  END AS quarantine_reason
FROM datasheet_project.silver.datasheets_clean

In [0]:
%sql
SELECT 
  chip_name,
  flash_kb,
  ram_kb,
  voltage_min,
  voltage_max,
  max_clock_mhz,
  pin_count,
  quarantine_reason
FROM datasheet_project.silver.datasheets_quarantine
WHERE is_quarantined = TRUE

In [0]:
%sql
-- Clean records → feed analytics/dashboards
CREATE OR REPLACE VIEW datasheet_project.gold.chip_catalog_clean AS
SELECT * FROM datasheet_project.silver.datasheets_quarantine
WHERE is_quarantined = FALSE;

-- Failed records → remediation
CREATE OR REPLACE VIEW datasheet_project.gold.chip_catalog_quarantine AS
SELECT * FROM datasheet_project.silver.datasheets_quarantine
WHERE is_quarantined = TRUE;

In [0]:
%sql
SELECT 'clean' AS layer, COUNT(*) AS cnt FROM datasheet_project.gold.chip_catalog_clean
UNION ALL
SELECT 'quarantine', COUNT(*) FROM datasheet_project.gold.chip_catalog_quarantine
UNION ALL
SELECT 'total', COUNT(*) FROM datasheet_project.silver.datasheets_quarantine